<a href="https://colab.research.google.com/github/Kalyan180655/bayesian-optimization-capstone/blob/main/Bayesian_Optimization_Capstone_Master.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Black-Box Optimisation (BBO) Capstone Project
**Cohort:** IMP-PCMLAI-26-06  
**Author:** Kalyan Logisetti  

## Overview
This notebook implements an iterative Bayesian Optimisation framework across 8 synthetic black-box functions ranging from 2D to 8D. Each weekly iteration fits Gaussian Process (GP) surrogates using Matérn kernels with Automatic Relevance Determination (ARD) and evaluates an Upper Confidence Bound (UCB) acquisition function to select new query vectors.

## 1. Environment Setup & Data Loading
In this section, we import the required Python libraries, extract the initial dataset archive (`Initial_data_points_starter.zip`), and load the observations for all 8 synthetic functions into memory.

In [17]:
!unzip -o Initial_data_points_starter.zip

Archive:  Initial_data_points_starter.zip
  inflating: __MACOSX/._initial_data  
  inflating: __MACOSX/initial_data/._function_4  
  inflating: __MACOSX/initial_data/._function_2  
  inflating: __MACOSX/initial_data/._function_3  
  inflating: __MACOSX/initial_data/._function_1  
  inflating: __MACOSX/initial_data/._.DS_Store  
  inflating: __MACOSX/initial_data/._function_7  
  inflating: __MACOSX/initial_data/._function_8  
  inflating: __MACOSX/initial_data/._function_6  
  inflating: __MACOSX/initial_data/._function_5  
  inflating: initial_data/function_5/initial_outputs.npy  
  inflating: initial_data/function_5/initial_inputs.npy  
  inflating: initial_data/function_8/initial_outputs.npy  
  inflating: initial_data/function_8/initial_inputs.npy  
  inflating: initial_data/function_7/initial_outputs.npy  
  inflating: initial_data/function_7/initial_inputs.npy  
  inflating: initial_data/function_1/initial_outputs.npy  
  inflating: initial_data/function_1/initial_inputs.npy  
  

In [18]:
import os
import numpy as np
import pandas as pd

# Define expected dimensions for Functions 1 through 8
FUNCTION_SPECS = {
    "Function 1": 2,
    "Function 2": 2,
    "Function 3": 3,
    "Function 4": 4,
    "Function 5": 4,
    "Function 6": 5,
    "Function 7": 6,
    "Function 8": 8,

}

# Dictionary to hold the loaded arrays
data_dict = {}

# Load starter data for each function subfolder
for i in range(1, 9):
    func_key = f"Function {i}"
    x_path = f"initial_data/function_{i}/initial_inputs.npy"
    y_path = f"initial_data/function_{i}/initial_outputs.npy"

    X = np.load(x_path)
    y = np.load(y_path)

    data_dict[func_key] = {
        "X": X,
        "y": y
    }

# Display summary table of loaded datasets
summary_data = []
for func_name, spec in FUNCTION_SPECS.items():
    X_shape = data_dict[func_name]["X"].shape
    y_shape = data_dict[func_name]["y"].shape
    summary_data.append({
        "Function": func_name,
        "Expected Dim": f"{spec}D",
        "X Shape (Inputs)": str(X_shape),
        "y Shape (Outputs)": str(y_shape),
        "Sample Input (Row 0)": np.array2string(data_dict[func_name]["X"][0], precision=4, suppress_small=True)
    })

summary_df = pd.DataFrame(summary_data)
print("Data Loading Complete. Summary Table:")
summary_df

Data Loading Complete. Summary Table:


,Function,Expected Dim,X Shape (Inputs),y Shape (Outputs),Sample Input (Row 0)
0,Function 1,2D,"(10, 2)","(10,)",[0.3194 0.763 ]
1,Function 2,2D,"(10, 2)","(10,)",[0.6658 0.124 ]
2,Function 3,3D,"(15, 3)","(15,)",[0.1715 0.3439 0.2487]
3,Function 4,4D,"(30, 4)","(30,)",[0.897 0.7256 0.1754 0.7017]
4,Function 5,4D,"(20, 4)","(20,)",[0.1914 0.0382 0.6074 0.4146]
5,Function 6,5D,"(20, 5)","(20,)",[0.7282 0.1547 0.7326 0.694 0.0564]
6,Function 7,6D,"(30, 6)","(30,)",[0.2726 0.3245 0.8971 0.833 0.1541 0.7959]
7,Function 8,8D,"(40, 8)","(40,)",[0.605 0.2922 0.9085 0.3555 0.2017 0.5753 0.3...


In [19]:
# Week 1 observations received via email
week1_inputs = {
    "Function 1": [0.731024, 0.733000],
    "Function 2": [0.702637, 0.926564],
    "Function 3": [0.492581, 0.611593, 0.340176],
    "Function 4": [0.577766, 0.428772, 0.425826, 0.249007],
    "Function 5": [0.224189, 0.846480, 0.879484, 0.878516],
    "Function 6": [0.728186, 0.154693, 0.732552, 0.693997, 0.056401],
    "Function 7": [0.057896, 0.491672, 0.247422, 0.218118, 0.420428, 0.730970],
    "Function 8": [0.056447, 0.065956, 0.022929, 0.038786, 0.403935, 0.801055, 0.488307, 0.893085],
}

week1_outputs = {
    "Function 1": 7.709397204501192e-16,
    "Function 2": 0.608464993364215,
    "Function 3": -0.03818251825730173,
    "Function 4": -4.02555701687789,
    "Function 5": 1088.8581918803375,
    "Function 6": -0.7046328290579386,
    "Function 7": 1.3649700705707235,
    "Function 8": 9.598482138169,
}

# Append Week 1 data into data_dict
for func_name in FUNCTION_SPECS.keys():
    new_x = np.array(week1_inputs[func_name]).reshape(1, -1)
    new_y = np.array([week1_outputs[func_name]])

    data_dict[func_name]["X"] = np.vstack([data_dict[func_name]["X"], new_x])
    data_dict[func_name]["y"] = np.append(data_dict[func_name]["y"], new_y)

# Print updated sample counts to confirm
print("=== UPDATED DATASET SIZES ===")
for func_name in FUNCTION_SPECS.keys():
    print(f"{func_name}: X shape = {data_dict[func_name]['X'].shape}, y shape = {data_dict[func_name]['y'].shape}")

=== UPDATED DATASET SIZES ===
Function 1: X shape = (11, 2), y shape = (11,)
Function 2: X shape = (11, 2), y shape = (11,)
Function 3: X shape = (16, 3), y shape = (16,)
Function 4: X shape = (31, 4), y shape = (31,)
Function 5: X shape = (21, 4), y shape = (21,)
Function 6: X shape = (21, 5), y shape = (21,)
Function 7: X shape = (31, 6), y shape = (31,)
Function 8: X shape = (41, 8), y shape = (41,)


## 2. Surrogate Modeling & Acquisition Strategy

To model the unknown black-box functions, we fit a **Gaussian Process (GP) Regressor** with a **Matérn 2.5 ARD kernel**. The Automatic Relevance Determination (ARD) allows the model to learn individual feature weights and length-scales across each dimension.

To guide the selection of the next evaluation points, we maximize the **Upper Confidence Bound (UCB)** acquisition function:

$$UCB(x) = \mu(x) + \beta \cdot \sigma(x)$$

Where:
- $\mu(x)$ represents predicted mean reward (Exploitation).
- $\sigma(x)$ represents predicted model uncertainty/variance (Exploration).
- $\beta = 2.0$ controls the exploration-exploitation balance.

We optimize $-UCB(x)$ using multi-start **L-BFGS-B** within the bounded hypercube domain $[0, 1]^d$.

In [20]:
# Multi-start L-BFGS-B optimization for UCB Acquisition across all 8 functions

from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, ConstantKernel as C
from scipy.optimize import minimize

def generate_weekly_queries(data_dict, beta=2.0, random_state=42):
    np.random.seed(random_state)
    query_results = {}

    for func_name, dim in FUNCTION_SPECS.items():
        X = np.array(data_dict[func_name]['X'])
        y = np.array(data_dict[func_name]['y']).ravel()

        # 1. Fit Gaussian Process with Matérn 2.5 ARD Kernel
        kernel = C(1.0, (1e-3, 1e3)) * Matern(length_scale=np.ones(dim), length_scale_bounds=(1e-2, 1e2), nu=2.5)
        gp = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=3, random_state=random_state)
        gp.fit(X, y)

        # 2. Define Negative UCB Acquisition Function
        def neg_ucb(x_vec):
            x_vec = np.atleast_2d(x_vec)
            mean, std = gp.predict(x_vec, return_std=True)
            return -(mean[0] + beta * std[0])

        # 3. Multi-start L-BFGS-B Optimization
        best_x = None
        best_val = np.inf
        bounds = [(0.0, 1.0)] * dim

        for _ in range(15):
            x0 = np.random.uniform(0.0, 1.0, dim)
            res = minimize(neg_ucb, x0, bounds=bounds, method='L-BFGS-B')
            if res.fun < best_val:
                best_val = res.fun
                best_x = res.x

        # 4. Format query as hyphen-separated 6-decimal float string
        formatted_str = "-".join([f"{val:.6f}" for val in best_x])
        query_results[func_name] = formatted_str
        print(f"{func_name} ({dim}D): {formatted_str}")

    return query_results

print("--- GENERATING WEEK 2 QUERY CANDIDATES ---")
week2_queries = generate_weekly_queries(data_dict, beta=2.0)

--- GENERATING WEEK 2 QUERY CANDIDATES ---
Function 1 (2D): 1.000000-1.000000
Function 2 (2D): 0.640966-0.143811
Function 3 (3D): 0.388677-0.271349-0.828738
Function 4 (4D): 0.412561-0.416447-0.353599-0.427238
Function 5 (4D): 0.447589-0.781334-0.893592-0.872509
Function 6 (5D): 0.243990-0.973011-0.393098-0.892047-0.631139
Function 7 (6D): 0.000000-0.322382-0.795014-0.160164-0.341980-0.694680
Function 8 (8D): 0.000000-0.000000-0.107312-0.000000-1.000000-0.272941-0.000000-0.000000


In [21]:
import warnings
from sklearn.exceptions import ConvergenceWarning

# Suppress hyperparameter boundary convergence warnings for clean presentation
warnings.filterwarnings("ignore", category=ConvergenceWarning)

# Format into a clean Pandas DataFrame for submission/portfolio display
results_df = pd.DataFrame([
    {"Function": name, "Dimension": f"{FUNCTION_SPECS[name]}D", "Week 2 Query Point": query}
    for name, query in week2_queries.items()
])

print("=== WEEK 2 SUBMISSION QUERIES ===")
results_df

=== WEEK 2 SUBMISSION QUERIES ===


,Function,Dimension,Week 2 Query Point
0,Function 1,2D,1.000000-1.000000
1,Function 2,2D,0.640966-0.143811
2,Function 3,3D,0.388677-0.271349-0.828738
3,Function 4,4D,0.412561-0.416447-0.353599-0.427238
4,Function 5,4D,0.447589-0.781334-0.893592-0.872509
5,Function 6,5D,0.243990-0.973011-0.393098-0.892047-0.631139
6,Function 7,6D,0.000000-0.322382-0.795014-0.160164-0.341980-0...
7,Function 8,8D,0.000000-0.000000-0.107312-0.000000-1.000000-0...


### Strategy Reflection & Observations (Week 2)

- **Model Fitting:** Gaussian Process models were successfully fit across all 8 dimensions using Matérn 2.5 ARD kernels to handle varying feature sensitivity across parameters.
- **Exploration vs. Exploitation ($\beta = 2.0$):** The UCB acquisition function naturally prioritized regions with high epistemic uncertainty, driving several queries toward the domain boundaries ($0.0$ and $1.0$) where initial data points were sparse.
- **Next Steps:** Once the evaluations for these Week 2 query points are returned, we will append $(x_{new}, y_{new})$ to `data_dict` and update our GP surrogates for the Week 3 iteration.